# Verbal Fluency and Spatial Arrangement (SpAM) Analysis

This notebook presents descriptive and preliminary inferential analyses of
Verbal Fluency Task (VFT) and Spatial Arrangement Method (SpAM) data.

The analyses are designed to be multi-subject ready and follow a
visualization-first approach appropriate for Report 1.


In [1]:
import json
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")

lemmatizer = WordNetLemmatizer()

english_vocab = set()

for synset in wn.all_synsets():
    for lemma in synset.lemma_names():
        english_vocab.add(lemma.lower())

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kotad\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Visualization settings

In [3]:
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## Load and inspect raw data

In [4]:
with open("responses.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

type(raw)


dict

In [5]:
raw.keys()


dict_keys(['fluency-spam'])

In [6]:
exp = raw["fluency-spam"]
type(exp), len(exp)


(dict, 35)

In [7]:
rows = []

for session_id, participant in exp.items():
    subject_id = participant["subject_id"]
    trials = participant["data"]
    
    for trial in trials:
        trial_copy = trial.copy()
        trial_copy["subject_id"] = subject_id
        trial_copy["session_id"] = session_id
        rows.append(trial_copy)

len(rows)

949

In [8]:
df = pd.DataFrame(rows)
df.shape

(949, 38)

In [9]:
df.head(3)

,trial_type,trial_index,time_elapsed,internal_node_id,subject,subject_id,session_id,rt,stimulus,response,...,language_confidence,language_acquisition,state_ut,age,gender,education,question_order,dominant_hand,alert_time,other_info
0,call-function,0,2,0.0-0.0,10255,10255,1qmxoH7jT7VECeLUVEKU,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,html-button-response,2,253030,0.0-2.0,10255,10255,1qmxoH7jT7VECeLUVEKU,5128.0,"\n <p style=""font-size:20px...",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,html-button-response,3,261988,0.0-3.0,10255,10255,1qmxoH7jT7VECeLUVEKU,8956.0,"\n <p style=""font-size:22px...",0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df["trial_type"].value_counts()

trial_type
html-button-response      455
survey-html-form          210
html-keyboard-response    144
call-function              35
instructions               35
survey-multi-choice        35
survey-text                35
Name: count, dtype: int64

In [11]:
df["task"].value_counts(dropna=False)

task
NaN     665
VFT     144
SpAM    140
Name: count, dtype: int64

In [12]:
vft_trials = df[df["task"] == "VFT"].copy()
vft_trials.shape

(144, 38)

In [13]:
vft_trials[[
    "subject_id",
    "domain",
    "tagged_responses",
    "response_times"
]].head(3)

,subject_id,domain,tagged_responses,response_times
3,10255,furniture-practice,"[{""response"":""chair"",""tag"":1},{""response"":""sof...","[9613.2,1365.9,1445.4,3981.5,2808.5,8129.8,374..."
7,10255,colours,"[{""response"":""red"",""tag"":1},{""response"":""blue""...","[2558.3,1464.6,1505.6,1894.6,1583.3,2164.2,823..."
11,10255,animals,"[{""response"":""lion"",""tag"":1},{""response"":""tige...","[3114.2,1989.2,2666.4,3306.2,2273.4,1772.4,205..."


In [14]:
import ast

vft_rows = []

for _, row in vft_trials.iterrows():
    try:
        words = ast.literal_eval(row["tagged_responses"])
        rts = ast.literal_eval(row["response_times"])
    except Exception as e:
        print("Parsing error for subject", row["subject_id"], e)
        continue

    # Safety check
    if len(words) != len(rts):
        print("Length mismatch for subject", row["subject_id"], row["domain"])
        continue

    for idx, (w, rt) in enumerate(zip(words, rts), start=1):
        vft_rows.append({
            "subject_id": row["subject_id"],
            "session_id": row["session_id"],
            "domain": row["domain"],
            "word": w["response"].lower().strip(),
            "rt_ms": float(rt),
            "position": idx
        })

In [15]:
vft_df = pd.DataFrame(vft_rows)
vft_df.shape

(1232, 6)

In [16]:
vft_df.isna().sum()

subject_id    0
session_id    0
domain        0
word          0
rt_ms         0
position      0
dtype: int64

In [17]:
vft_df["domain"].value_counts()

domain
animals               365
foods                 326
body-parts            206
furniture-practice    188
colours               147
Name: count, dtype: int64

In [18]:
vft_df.groupby(["subject_id", "domain"])["position"].max().describe()

count    140.000000
mean       8.800000
std        4.345178
min        1.000000
25%        6.000000
50%        8.000000
75%       11.000000
max       24.000000
Name: position, dtype: float64

In [19]:
vft_df_clean = vft_df[~vft_df["domain"].str.contains("practice", case=False)].copy()
vft_df_clean["domain"].value_counts()

domain
animals       365
foods         326
body-parts    206
colours       147
Name: count, dtype: int64

In [20]:
import re

def classify_language(text):
    text_clean = re.sub(r"[^\w\s]", "", text).lower().strip()
    if text_clean == "":
        return "Other"

    for char in text_clean:
        if '\u0900' <= char <= '\u097F':
            return "Hindi/Hinglish"

    tokens = text_clean.split()
    english_count = 0
    for token in tokens:
        if token in english_vocab:
            english_count += 1
    if english_count >= len(tokens) / 2:
        return "English"
    return "Hindi/Hinglish"

In [21]:
vft_df_clean["language_type"] = vft_df_clean["word"].apply(classify_language)

In [22]:
vft_df_clean

,subject_id,session_id,domain,word,rt_ms,position,language_type
9,10255,1qmxoH7jT7VECeLUVEKU,colours,red,2558.3,1,English
10,10255,1qmxoH7jT7VECeLUVEKU,colours,blue,1464.6,2,English
11,10255,1qmxoH7jT7VECeLUVEKU,colours,green,1505.6,3,English
12,10255,1qmxoH7jT7VECeLUVEKU,colours,indigo,1894.6,4,English
13,10255,1qmxoH7jT7VECeLUVEKU,colours,orange,1583.3,5,English
...,...,...,...,...,...,...,...
1227,42616,z5YTCkkdyK0TXu4JiPmY,animals,भैंस,7745.2,2,Hindi/Hinglish
1228,42616,z5YTCkkdyK0TXu4JiPmY,animals,बैल,7655.9,3,Hindi/Hinglish
1229,42616,z5YTCkkdyK0TXu4JiPmY,animals,बकरी,7831.5,4,Hindi/Hinglish
1230,42616,z5YTCkkdyK0TXu4JiPmY,animals,घोड़ा,8554.0,5,Hindi/Hinglish


In [23]:
vft_df_clean.to_csv("vft_responses.csv", index=False)

In [24]:
spam_trials = df[df["task"] == "SpAM"].copy()

spam_trials = spam_trials[[
    "subject_id",
    "session_id",
    "domain",
    "droppedwords"
]]

spam_trials.head()


,subject_id,session_id,domain,droppedwords
5,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,"[{'word': 'chair', 'id': 'word-1', 'x_px': 702..."
9,10255,1qmxoH7jT7VECeLUVEKU,colours,"[{'word': 'red', 'id': 'word-1', 'x_px': 94.20..."
13,10255,1qmxoH7jT7VECeLUVEKU,animals,"[{'word': 'lion', 'id': 'word-1', 'x_px': 32.1..."
17,10255,1qmxoH7jT7VECeLUVEKU,foods,"[{'word': 'lays', 'id': 'word-1', 'x_px': 121...."
32,95712,2kupFqWyVgUTsqisMsI7,furniture-practice,"[{'word': 'टेबल', 'id': 'word-1', 'x_px': 22, ..."


In [25]:
spam_rows = []

for _, row in spam_trials.iterrows():
    if not isinstance(row["droppedwords"], list):
        continue

    for w in row["droppedwords"]:
        spam_rows.append({
            "subject_id": row["subject_id"],
            "session_id": row["session_id"],
            "domain": row["domain"],
            "word": w.get("word"),
            "x": w.get("x_norm"),
            "y": w.get("y_norm"),
        })

spam_df = pd.DataFrame(spam_rows)
spam_df.head()


,subject_id,session_id,domain,word,x,y
0,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,chair,0.574602,1.003870
1,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,sofa,0.498202,0.996904
2,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,bench,0.349398,0.989164
3,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,lights,0.982554,0.652477
4,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,dinig table,0.674591,1.003870


In [26]:
print(spam_df.shape)
print(spam_df.isna().sum())
spam_df["domain"].value_counts()


(1959, 6)
subject_id    0
session_id    0
domain        0
word          0
x             0
y             0
dtype: int64


domain
animals               581
foods                 506
body-parts            337
furniture-practice    298
colours               237
Name: count, dtype: int64

In [29]:
spam_df_clean = spam_df[~spam_df["domain"].str.contains("practice", case=False)].copy()
spam_df_clean["domain"].value_counts()


domain
animals       581
foods         506
body-parts    337
colours       237
Name: count, dtype: int64

In [30]:
print(spam_df_clean.shape)

(1661, 6)
